In [ ]:

import cv2
import os
from pathlib import Path


BASE_DIR = Path(r"C:\Users\PC\OneDrive\Documentos\tarea\semestre 9\IA\rc")
DATA_DIR = BASE_DIR / "fotos"

IMG_SIZE = 160
MAX_FOTOS = 200  

BASE_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# Detector HaarCascade
cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
if cascade.empty():
    raise RuntimeError("No se cargó el clasificador de rostros.")

# Nombre de persona
person_name = input("Nombre de la persona: ").strip()
person_dir = DATA_DIR / person_name
person_dir.mkdir(exist_ok=True)

print(f"\n Se guardarán fotos en: {person_dir}")
print(f"Tomará automáticamente hasta {MAX_FOTOS} fotos.")
print("Presiona ESC para salir.\n")


cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError("No se pudo abrir la cámara.")

img_count = 0

while True:
    ok, frame = cap.read()
    if not ok:
        print("⚠️ No se pudo leer frame.")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    
    faces = cascade.detectMultiScale(gray, 1.2, 5, minSize=(80, 80))

    
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)

        if img_count < MAX_FOTOS:
            face = frame[y:y+h, x:x+w]

            if face.size > 0:
                face_resized = cv2.resize(face, (IMG_SIZE, IMG_SIZE))

                filename = person_dir / f"{person_name}_{img_count:04d}.jpg"
                cv2.imwrite(str(filename), face_resized)

                img_count += 1
                print(f"✔️ Foto {img_count}/{MAX_FOTOS} guardada")

   
    cv2.putText(frame, f"{person_name} | Fotos: {img_count}/{MAX_FOTOS}",
                (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)

    cv2.imshow("Auto-Captura Rostros (ESC para salir)", frame)

  
    key = cv2.waitKey(1) & 0xFF
    if key == 27 or img_count >= MAX_FOTOS:
        break

cap.release()
cv2.destroyAllWindows()

print(f"\n📸 Captura finalizada. Total guardadas: {img_count}")
print(f"Guardadas en: {person_dir}")



📸 Se guardarán fotos en: C:\Users\PC\OneDrive\Documentos\tarea\semestre 9\IA\rc\fotos\Axel Alarcon
Tomará automáticamente hasta 200 fotos.
Presiona ESC para salir.

✔️ Foto 1/200 guardada
✔️ Foto 2/200 guardada
✔️ Foto 3/200 guardada
✔️ Foto 4/200 guardada
✔️ Foto 5/200 guardada
✔️ Foto 6/200 guardada
✔️ Foto 7/200 guardada
✔️ Foto 8/200 guardada
✔️ Foto 9/200 guardada
✔️ Foto 10/200 guardada
✔️ Foto 11/200 guardada
✔️ Foto 12/200 guardada
✔️ Foto 13/200 guardada
✔️ Foto 14/200 guardada
✔️ Foto 15/200 guardada
✔️ Foto 16/200 guardada
✔️ Foto 17/200 guardada
✔️ Foto 18/200 guardada
✔️ Foto 19/200 guardada
✔️ Foto 20/200 guardada
✔️ Foto 21/200 guardada
✔️ Foto 22/200 guardada
✔️ Foto 23/200 guardada
✔️ Foto 24/200 guardada
✔️ Foto 25/200 guardada
✔️ Foto 26/200 guardada
✔️ Foto 27/200 guardada
✔️ Foto 28/200 guardada
✔️ Foto 29/200 guardada
✔️ Foto 30/200 guardada
✔️ Foto 31/200 guardada
✔️ Foto 32/200 guardada
✔️ Foto 33/200 guardada
✔️ Foto 34/200 guardada
✔️ Foto 35/200 guardada
✔️ 

In [2]:

import os
import json
from pathlib import Path       
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model

print("TensorFlow:", tf.__version__)


BASE_DIR = Path(r"C:\Users\PC\OneDrive\Documentos\tarea\semestre 9\IA\rc")
DATA_DIR = BASE_DIR / "fotos"
MODEL_PATH = BASE_DIR / "modelo_cnn_rostros.keras"
LABELS_PATH = BASE_DIR / "labels_indices.json"

IMG_SIZE = 160


classes = [d.name for d in DATA_DIR.iterdir() if d.is_dir()]
print("Clases encontradas:", classes)

if len(classes) < 2:
    raise RuntimeError("Se necesitan al menos 2 clases para entrenar. Captura más rostros en CELDA 1.")

BATCH_SIZE = 16
EPOCHS = 15


train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.2,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

valid_generator = valid_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)


with open(LABELS_PATH, "w") as f:
    json.dump(train_generator.class_indices, f, indent=4)

print("Índices guardados:", train_generator.class_indices)


base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(len(classes), activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()



history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=valid_generator
)

model.save(MODEL_PATH)
print("Modelo guardado en:", MODEL_PATH)


TensorFlow: 2.16.1
Clases encontradas: ['Axel Alarcon', 'chester bennington', 'dwayne johnson', 'Fernando Rosales', 'Jimena De Los Rios', 'lopez doriga', 'luis miguel', 'Ryan Gosling', 'Sekisaka Tajin', 'shakira', 'will smith']
Found 2395 images belonging to 11 classes.
Found 598 images belonging to 11 classes.
Índices guardados: {'Axel Alarcon': 0, 'Fernando Rosales': 1, 'Jimena De Los Rios': 2, 'Ryan Gosling': 3, 'Sekisaka Tajin': 4, 'chester bennington': 5, 'dwayne johnson': 6, 'lopez doriga': 7, 'luis miguel': 8, 'shakira': 9, 'will smith': 10}


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 160, 160,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 80, 80,    │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 80, 80,    │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 80, 80,    │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 80, 80,    │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 80, 80,    │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 80, 80,    │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 80, 80,    │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 80, 80,    │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 80, 80,    │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 80, 80,    │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 80, 80,    │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 81, 81,    │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 40, 40,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 40, 40,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 40, 40,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 40, 40,    │      2,304 │ block_1_depthwis

 Total params: 2,423,371 (9.24 MB)

 Trainable params: 165,387 (646.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 38s 236ms/step - accuracy: 0.9148 - loss: 0.3190 - val_accuracy: 0.9950 - val_loss: 0.0141
Epoch 2/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 15s 100ms/step - accuracy: 0.9942 - loss: 0.0255 - val_accuracy: 0.9950 - val_loss: 0.0138
Epoch 3/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 16s 105ms/step - accuracy: 0.9962 - loss: 0.0167 - val_accuracy: 0.9983 - val_loss: 0.0138
Epoch 4/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 15s 97ms/step - accuracy: 0.9975 - loss: 0.0107 - val_accuracy: 0.9950 - val_loss: 0.0129
Epoch 5/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 14s 96ms/step - accuracy: 0.9979 - loss: 0.0093 - val_accuracy: 0.9950 - val_loss: 0.0197
Epoch 6/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 15s 97ms/step - accuracy: 0.9987 - loss: 0.0039 - val_accuracy: 0.9950 - val_loss: 0.0142
Epoch 7/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 14s 96ms/step - accuracy: 0.9979 - loss: 0.0082 - val_accuracy: 0.9950 - val_loss: 0.0148
Epoch 8/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 14s 94ms/step - accuracy: 0.9971 - loss: 0.0067

In [3]:

import cv2
import numpy as np
from pathlib import Path
import json
from tensorflow.keras.models import load_model


BASE_DIR = Path(r"C:\Users\PC\OneDrive\Documentos\tarea\semestre 9\IA\rc")
MODEL_PATH = BASE_DIR / "modelo_cnn_rostros.keras"
LABELS_PATH = BASE_DIR / "labels_indices.json"
IMG_SIZE = 160

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"No se encontró el modelo en {MODEL_PATH}. Entrena el modelo primero.")

model = load_model(MODEL_PATH)
print("Modelo cargado desde:", MODEL_PATH)


if not LABELS_PATH.exists():
    raise FileNotFoundError("No se encontró labels_indices.json.")

with open(LABELS_PATH, "r") as f:
    class_indices = json.load(f)

idx_to_class = {v: k for k, v in class_indices.items()}
print("Clases:", idx_to_class)


cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

print("\nReconocimiento iniciado. Presiona ESC para salir.")

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = cascade.detectMultiScale(gray, 1.2, 5, minSize=(80, 80))

    for (x, y, w, h) in faces:
        face = frame[y:y+h, x:x+w]
        face_resized = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
        face_array = face_resized.astype("float32") / 255.0
        face_array = np.expand_dims(face_array, axis=0)

        preds = model.predict(face_array, verbose=0)[0]
        idx = np.argmax(preds)
        prob = preds[idx]
        label = idx_to_class[idx]

        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, f"{label} ({prob*100:.1f}%)", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    cv2.imshow("Reconocimiento facial", frame)

    if cv2.waitKey(1) & 0xFF == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()


Modelo cargado desde: C:\Users\PC\OneDrive\Documentos\tarea\semestre 9\IA\rc\modelo_cnn_rostros.keras
Clases: {0: 'Axel Alarcon', 1: 'Fernando Rosales', 2: 'Jimena De Los Rios', 3: 'Ryan Gosling', 4: 'Sekisaka Tajin', 5: 'chester bennington', 6: 'dwayne johnson', 7: 'lopez doriga', 8: 'luis miguel', 9: 'shakira', 10: 'will smith'}

Reconocimiento iniciado. Presiona ESC para salir.
